# Welcome to the `go` model linear programming tutorial!

In [1]:
import go
from go import Model


In [2]:
f = "/Users/d3y010/Desktop/config.yml"


In [3]:
# go.build_data_file(config_file=f)


In [4]:
model = Model(region="west", problem="linear", complexity="multi")


In [5]:
model.run(config_file=f, n_days=1)


AttributeError: 'str' object has no attribute 'is_constructed'

In [8]:
config = go.generate_config(config_file=f)


In [5]:
config.dat_file

'WECC_data.dat'

In [1]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory



In [30]:

class GoSolver:

    SUPPORTED = ("appsi_highs", "cplex", "gurobi")

    def __init__(self, solver_name, **kwargs):
        
        self.solver_name = self.validate_name(solver_name)
        self.go_solver = SolverFactory(self.solver_name)
        self.params = kwargs
        
        # apply options
        self.option_generator()

    @staticmethod
    def validate_name(solver_name):
        """Validate solver name."""

        if solver_name not in GoSolver.SUPPORTED:
            raise KeyError(
                f"Solver '{solver_name}' is not currently supported.  Use 'appsi_highs', 'cplex', or 'gurobi'"
            )

        return solver_name

    def option_generator(self):
        """Apply options based on solver and user desired parameters."""

        if self.solver_name == "appsi_highs":
            self.set_highs_options()
            
        elif self.solver_name == "cplex":
            self.set_cplex_options()
            
        elif self.solver_name == "gurobi":
            self.set_gurobi_options()

    def set_highs_options(self):
        """Setup HiGHS options.  Use defaults of no parameter values are specified.

        See full list here:  https://ergo-code.github.io/HiGHS/stable/options/definitions/#option-definitions
        """
        
        self.go_solver.options["presolve"] = self.params.get("presolve", "choose")
        self.go_solver.options["solver"] = self.params.get("solver", "simplex")
        self.go_solver.options["parallel"] = self.params.get("parallel", "on")
        self.go_solver.options["run_crossover"] = self.params.get("run_crossover", "on")
        self.go_solver.options["time_limit"] = self.params.get("time_limit", 3600)
        self.go_solver.options["threads"] = self.params.get("threads", 8)
        self.go_solver.options["simplex_strategy"] = self.params.get("simplex_strategy", 2)  # dual simplex

    def set_cplex_options(self):
        """Setup CPLEX options. Use defaults of no parameter values are specified."""

        pass

    def set_gurobi_options(self):
        """Setup GUROBI options. Use defaults of no parameter values are specified."""

        pass


In [31]:
g = GoSolver(solver_name="appsi_highs")



highs


In [32]:
g.go_solver.options

{'presolve': 'choose',
 'solver': 'simplex',
 'parallel': 'on',
 'run_crossover': 'on',
 'time_limit': 3600,
 'threads': 8,
 'simplex_strategy': 2}

## A brief introduction

We will be running a linear programming model for the Western Electricity Coordinating Council (WECC) region.

## Install `go` from GitHub

```bash
python -m pip install -e git://github.com/IMMM-SFA/go.git@main#egg=go

```

## Load packages

In [ ]:
import pyomo.environ as pyo

from go.wecc import LinearProgrammingModel
# from go.wecc import MixedIntegerModel


## Illustrative subset of data

In [ ]:
# test WECC data
wecc_data = '/Users/d3y010/repos/github/IM3_WECC/Model/WECC_data.dat'


## Create .dat file

This file is created by another code that combines inputs of different CSV files.

From this script:  `/Users/d3y010/repos/github/IM3_WECC/Model/WECCDataSetup.py`

Explore this one to see contents `/Users/d3y010/repos/github/IM3_WECC/Model/UC.dat`



## Run linear programming model

In [ ]:
model = LinearProgrammingModel(data=wecc_data)

model


In [ ]:
inst = model.create_instance()


## Setup the solver

In [ ]:
opt = pyo.SolverFactory('glpk')


## Need to download `cbc` executable

Look into threading:  `opt.solve(instance, options={"threads": 4})`


In [ ]:
opt.solve(inst, tee=True)


In [ ]:
inst.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)


In [ ]:
days = 4

H = inst.HorizonHours
D = 2
K=range(1,H+1)


In [ ]:
for day in range(1, days):
    
    for z in inst.busses:
        
        # load demand and reserve time series data
        for i in K:
            
            inst.HorizonDemand[z, i] = instance.SimDemand[z, (day-1) * 24 + i]
            
            
    for z in inst.Hydro:
        
        # load hydropower time series data
        inst.HorizonHydro[z] = instance.SimHydro[z, day]
        

    for z in inst.Solar:
        
        # load solar time series data
        inst.HorizonSolar[z, i] = instance.SimSolar[z, (day-1) * 24 + i]
        
    for z in inst.Wind:
        
        # load wind time series data
        inst.HorizonWind[z, i] = instance.SimWind[z, (day-1) * 24 + i]
        
    result = opt.solve(instance,tee=True,symbolic_solver_labels=True) ##,tee=True to check number of variables\n",
    instance.solutions.load_from(result) 
        
        
        
        
        